In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

import os
import glob

# Use the Java/Hadoop installation configured in your environment.
# Use the Java/Hadoop installation configured in your environment.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('TLC_FHVHV_Processing') \
    .master('local[*]') \
    .config("spark.driver.memory", "8g") \
    .config('spark.sql.parquet.enableVectorizedReader', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

print(f'Spark version: {spark.version}')
print('Environment ready!')

    # .config("spark.driver.maxResultSize", "3g") \
    # .config("spark.sql.shuffle.partitions", "32") \
    # .config("spark.memory.fraction", "0.8") \
    # .config("spark.memory.storageFraction", "0.2") \

# 02 — Data Processing & Aggregation: FHVHV Trip Records (2024–2025)

**Project:** NYC TLC Ridesharing Analysis Pipeline  
**Phase:** 1 — Data Foundation  
**Goal:** Clean raw FHVHV data in memory, engineer features, and aggregate to output files

**Local directories:** configure paths in the first code cell.
**Local directories:** configure paths in the first code cell.

---

### What this notebook does:
1. Set up PySpark environment
2. Load raw FHVHV parquet files (2024–2025)
3. Clean data (nulls, invalid values, outliers)
4. Engineer features (datetime, fare, wait time columns)
5. Aggregate to 5 output files:
   - `fhvhv_by_zone_month.parquet` — all metrics by zone × month
   - `fhvhv_by_hour.parquet` — all metrics by date × hour
   - `fhvhv_by_zone_hour.parquet` — all metrics by zone × date × hour
   - `fhvhv_by_15min.parquet` — wait time by date × 15min interval
   - `fhvhv_by_zone_15min.parquet` — wait time by zone × date × 15min interval

## 0. Environment Setup

## 1. Define Paths

In [ ]:
RAW_DATA_PATH = str(RAW_DIR)
OUTPUT_PATH = str(PROCESSED_DIR)

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Policy cutoff date
POLICY_DATE = '2025-01-05'

# Load 2024-2025 parquet files
TARGET_YEARS = ['2024', '2025']
all_files = glob.glob(os.path.join(RAW_DATA_PATH, 'fhvhv_*.parquet'))
target_files = sorted([
    f for f in all_files
    if any(year in os.path.basename(f) for year in TARGET_YEARS)
])

print(f'Files to load: {len(target_files)}')
print(f'Output path:   {OUTPUT_PATH}')

## 2. Load Raw Data

In [ ]:
print('Loading raw FHVHV parquet files...')
# df = spark.read.parquet(*target_files)


df = None
for file in target_files:
    temp_df = spark.read.parquet(file)

    if df is None:
        df = temp_df
    else:
        df = df.unionByName(temp_df,allowMissingColumns=True)
    print(f"loaded:{file} | Rows: {temp_df.count()}")

raw_count = df.count()
print(f'Raw rows loaded: {raw_count:,}')

## 3. Data Cleaning

Rules:
- Check the times for all records with null `cbd_congestion_fee` and fill null with 0 if they are during the pre-policy period (per-policy is not implemented then)
- Remove 16 duplicates
- Remove rows where `trip_miles <= 0` (invalid distance)
- Remove rows where `trip_time <= 0` (invalid duration)
- Remove rows where `base_passenger_fare <= 0` (invalid fare)
- Remove rows where `pickup_datetime < request_datetime` (impossible)
- Remove extreme outliers: `trip_miles` > 200, `trip_time` > 10800 (3 hours), `speed` > 120, or `speed` < 1

In [ ]:
df.printSchema()

In [ ]:
df = df\
    .withColumn('trip_time_cal',F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')) \
    .withColumn('trip_speed',(F.col('trip_miles')/F.col('trip_time'))*3600)

In [ ]:
# # check in records with null cbd_congestion_price 

# print('Null values after policy:',df.filter((F.col('cbd_congestion_fee').isNull()) & (F.col('pickup_datetime') >= '2025-01-05')).count())

# print('None-null values before policy:', df.filter((F.col('cbd_congestion_fee').isNotNull()) & (F.col('pickup_datetime') < '2025-01-05')).count())

# # Check the actual earliest date
# df.filter(F.col('cbd_congestion_fee').isNotNull()) \
#   .agg(F.min('pickup_datetime').alias('first_congestion_charge')) \
#   .show(truncate=False)

In [ ]:
# # Verify the exact cutoff with a few days around Jan 1

# df.filter(F.col('cbd_congestion_fee').isNotNull()) \
#   .withColumn('date', F.to_date('pickup_datetime')) \
#   .groupBy('date') \
#   .count() \
#   .orderBy('date') \
#   .show(20)

In [ ]:
df_clean = df \
    .dropDuplicates() \
    .filter(F.col('trip_miles') > 0) \
    .filter(F.col('trip_time_cal') > 0) \
    .filter(F.col('base_passenger_fare') > 0) \
    .filter(F.col('trip_speed') > 1) \
    .filter(F.col('driver_pay') > 0) \
    .filter(F.col('pickup_datetime') >= F.col('request_datetime')) \
    .filter(F.col('trip_miles') <= 200) \
    .filter(F.col('trip_time_cal') <= 10800) \
    .filter(F.col('trip_speed') <= 120) \
    .withColumn('cbd_congestion_fee',
        F.when(F.col('cbd_congestion_fee').isNull(), 0)
         .otherwise(F.col('cbd_congestion_fee')))

print('Cleaning complete!')
# clean_count = df_clean.count()
# removed = raw_count - clean_count
# print(f'Rows after cleaning: {clean_count:,}')
# print(f'Rows removed:        {removed:,} ({removed/raw_count*100:.2f}%)')

## 4. Feature Engineering

In [ ]:
df_feat = df_clean \
    .withColumn('year',         F.year('pickup_datetime')) \
    .withColumn('month',        F.month('pickup_datetime')) \
    .withColumn('week',         F.weekofyear('pickup_datetime')) \
    .withColumn('date',         F.to_date('pickup_datetime')) \
    .withColumn('hour',         F.hour('pickup_datetime')) \
    .withColumn('do_hour', F.hour('dropoff_datetime'))\
    .withColumn('day_of_week',  F.dayofweek('pickup_datetime')) \
    .withColumn('is_weekend',   F.when(F.dayofweek('pickup_datetime').isin([1, 7]), 1).otherwise(0)) \
    .withColumn('15min_interval', (F.minute('pickup_datetime') / 15).cast('integer') * 15) \
    .withColumn('policy_period',
        F.when(F.col('pickup_datetime') < F.lit(POLICY_DATE), 'pre_policy')
         .otherwise('post_policy')) \
    .withColumn('wait_time_sec',
        F.unix_timestamp('pickup_datetime') - F.unix_timestamp('request_datetime')) \
    .withColumn('trip_duration_min', F.col('trip_time_cal') / 60) \
    .withColumn('total_fare',
        F.col('base_passenger_fare') + F.col('tolls') + F.col('tips') +
        F.col('congestion_surcharge') + F.col('sales_tax') + F.col('bcf') +
        F.col('airport_fee') + F.col('cbd_congestion_fee')) \
    .withColumn('tnc_comp',
        F.col('base_passenger_fare') + F.col('tips') - F.col('driver_pay'))

# Cache for repeated aggregations
df_feat.cache()
print('Feature engineering complete!')
new_cols = ['year', 'month', 'week', 'date', 'hour', 'day_of_week', 'is_weekend',
            '15min_interval', 'policy_period', 'wait_time_sec',
            'trip_duration_min', 'total_fare', 'tnc_comp']
print('New columns added:', new_cols)
# for c in new_cols:
#     print(f'  + {c}')

## 5. Define Aggregation Helper

Using aggregate-first approach:
- Sum numerators and denominators separately
- Then divide to get rates

In [ ]:
def compute_metrics(df, group_cols):
    """
    Aggregate all metrics using aggregate-first approach.
    Group by group_cols and compute all metrics.
    """
    return df.groupBy(group_cols).agg(
        # Demand
        F.count('*').alias('trip_count'),

        # Intermediate sums for rate calculations
        F.sum('wait_time_sec').alias('_sum_wait_time'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('driver_pay').alias('_sum_driver_pay'),
        F.sum('tnc_comp').alias('_sum_tnc_comp'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min')
    ).withColumn(
        'avg_wait_time_sec',     F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).withColumn(
        'base_fare_per_mile',    F.round(F.col('_sum_base_fare') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'base_fare_per_min',     F.round(F.col('_sum_base_fare') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'base_fare_per_trip',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'total_fare_per_trip',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'driver_pay_per_mile',   F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'driver_pay_per_min',    F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'driver_pay_per_trip',   F.round(F.col('_sum_driver_pay') / F.col('trip_count'), 4)
    ).withColumn(
        'tnc_commission_rate',   F.round(F.col('_sum_tnc_comp') / F.col('_sum_base_fare'), 4)
    ).drop(
        '_sum_wait_time', '_sum_base_fare', '_sum_total_fare',
        '_sum_driver_pay', '_sum_tnc_comp', '_sum_trip_miles', '_sum_trip_duration_min'
    )


def compute_waittime_only(df, group_cols):
    """
    Aggregate wait time only — used for 15min granularity files.
    """
    return df.groupBy(group_cols).agg(
        F.count('*').alias('trip_count'),
        F.sum('wait_time_sec').alias('_sum_wait_time')
    ).withColumn(
        'avg_wait_time_sec', F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).drop('_sum_wait_time')


print('Aggregation helpers defined!')

## 6. Aggregation & Save Output Files

### 6.1 fhvhv_by_zone_month.parquet
All metrics aggregated by `PULocationID × year × month`

In [ ]:
# print('Aggregating by zone × month...')

# for year in [2024, 2025]:
#     print(f'Processing year {year}...')
#     df_year = df_feat.filter(F.col('year') == year)
#     group_cols = ['PULocationID', 'year', 'month', 'policy_period']
#     df_zone_month = compute_metrics(df_year, group_cols) \
#         .orderBy('PULocationID', 'year', 'month')
    
#     output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_month_{year}.parquet')
#     df_zone_month.write.mode('overwrite').parquet(output_file)
#     print(f'Saved: {output_file}')
#     print(f'Rows:  {df_zone_month.count():,}')

### 6.2 fhvhv_by_hour.parquet
All metrics aggregated by `date × hour`

In [ ]:
print('Aggregating by date × hour...')

for year in [2024, 2025]:
    print(f"Processing year {year}...")
    df_year = df_feat.filter(F.col('year') == year)
    
    group_cols = ['date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period']
    df_hour = compute_metrics(df_year, group_cols).orderBy('date', 'hour')

    output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_hour_{year}.parquet')
    df_hour.write.mode('overwrite').parquet(output_file)
    print(f'Saved: {output_file}')

    # Read back to verify instead of counting before saving
    df_check = spark.read.parquet(output_file)
    print(f'Rows: {df_check.count():,}')
    df_check.show(5)
    
    # Free memory before next year
    df_year.unpersist()
    df_hour.unpersist()

### 6.3 fhvhv_by_zone_hour.parquet
All metrics aggregated by `PULocationID × date × hour`

In [ ]:
# plan a: focusing on all platforms 

print('Aggregating by zone × date × hour...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )
    
        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period']
        df_zone_hour = compute_metrics(df_month, group_cols).orderBy('PULocationID', 'date', 'hour')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_hour_{year}_{month:02d}.parquet')
        df_zone_hour.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_zone_hour.unpersist()

In [ ]:
# plan b: group by different platforms (Uber and Lyft)
import gc 

print('Aggregating by zone × date × hour...')


def is_complete(output_file):
    success_marker = os.path.join(output_file, '_SUCCESS')
    return os.path.exists(success_marker)


# # ── Create a filtered, persisted Uber-only dataset ──────────────────────────
# df_uber = df_feat.filter(F.col('hvfhs_license_num') == 'HV0003')
# df_uber.cache()
# df_uber.count()   # force materialization now, while the filter is fresh

# print(f'Uber dataset ready: {df_uber.count():,} rows')

# # Aggregate based on the uber dataset

for year in [2024,2025]:
    for month in range(1, 13):
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_hour_uber_{year}_{month:02d}.parquet')
        if is_complete(output_file):
            print(f'Already complete, skipping: {output_file}')
            continue
        
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month) & (F.col('hvfhs_license_num') == 'HV0003')
        )
    
        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period', 'hvfhs_license_num']

        df_agg = compute_metrics(df_month, group_cols)
        df_agg = df_agg.orderBy('PULocationID', 'date', 'hour')

        df_agg.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_month.unpersist()
        df_agg.unpersist()
        spark.catalog.clearCache()
        gc.collect()

In [ ]:
# plan b: group by different platforms (Uber and Lyft)
import gc 

print('Aggregating by zone × date × hour...')


def is_complete(output_file):
    success_marker = os.path.join(output_file, '_SUCCESS')
    return os.path.exists(success_marker)

# Aggregate based on the Lyft dataset

for year in [2024,2025]:
    for month in range(1, 13):
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_hour_Lyft_{year}_{month:02d}.parquet')
        if is_complete(output_file):
            print(f'Already complete, skipping: {output_file}')
            continue
        
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month) & (F.col('hvfhs_license_num') == 'HV0005')
        )
    
        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period', 'hvfhs_license_num']

        df_agg = compute_metrics(df_month, group_cols)
        df_agg = df_agg.orderBy('PULocationID', 'date', 'hour')

        df_agg.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_month.unpersist()
        df_agg.unpersist()
        spark.catalog.clearCache()
        gc.collect()

In [ ]:
df_test = df_feat.filter(
    (F.col('year') == 2024) & (F.col('month') == 1) & (F.col('hvfhs_license_num') == 'HV0003')
)
print(df_test.count())

### 6.4 fhvhv_by_15min.parquet
Wait time aggregated by `date × 15min interval`

In [ ]:
print('Aggregating wait time by date × 15min interval...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )
        group_cols = ['date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', '15min_interval', 'policy_period']
        df_15min = compute_waittime_only(df_month, group_cols).orderBy('date', 'hour', '15min_interval')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_15min_{year}_{month:02d}.parquet')
        df_15min.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_15min.unpersist()

### 6.5 fhvhv_by_zone_15min.parquet
Wait time aggregated by `PULocationID × date × 15min interval`

In [ ]:
print('Aggregating wait time by zone × date × 15min interval...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )

        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', '15min_interval', 'policy_period']
        df_zone_15min = compute_waittime_only(df_month, group_cols).orderBy('PULocationID', 'date', 'hour', '15min_interval')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_15min_{year}_{month:02d}.parquet')
        df_zone_15min.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_zone_15min.unpersist()

---
### 6.6 fhvhv_by_DOzone_month.parquet
All metrics aggregated by `DOLocationID × year × month`

> Mirror of 6.1 but on the **dropoff** side — needed for destination map in dashboard

In [ ]:
print('Aggregating by dropoff zone × month...')

for year in [2024, 2025]:
    print(f'Processing year {year}...')
    df_year = df_feat.filter(F.col('year') == year)

    group_cols = ['DOLocationID', 'year', 'month', 'policy_period']
    df_DO_zone_month = compute_metrics(df_year, group_cols) \
        .orderBy('DOLocationID', 'year', 'month')

    output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_DOzone_month_{year}.parquet')
    df_DO_zone_month.write.mode('overwrite').parquet(output_file)
    print(f'Saved: {output_file}')

    df_check = spark.read.parquet(output_file)
    print(f'Rows: {df_check.count():,}')
    df_check.show(5)

    df_year.unpersist()
    df_DO_zone_month.unpersist()

---
### 6.7 fhvhv_by_DO_hour.parquet
All metrics aggregated by `date × hour`

> Mirror of 6.2 but on the **dropoff** side — needed for destination map in dashboard

In [ ]:
print('Aggregating by dropoff zone × month...')

for year in [2024, 2025]:
    print(f'Processing year {year}...')
    df_year = df_feat.filter(F.col('year') == year)

    group_cols = ['date', 'year', 'month', 'day_of_week', 'is_weekend', 'do_hour', 'policy_period']
    df_DO_hour = compute_metrics(df_year, group_cols) \
        .orderBy('date', 'do_hour')

    output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_DO_hour_{year}.parquet')
    df_DO_hour.write.mode('overwrite').parquet(output_file)
    print(f'Saved: {output_file}')

    df_check = spark.read.parquet(output_file)
    print(f'Rows: {df_check.count():,}')
    df_check.show(5)

    df_year.unpersist()
    df_DO_hour.unpersist()

---
### 6.8 fhvhv_by_DOzone_hour.parquet
All metrics aggregated by `DOLocationID × date × hour` (monthly files)

> Mirror of 6.3 but on the **dropoff** side — needed for hourly destination patterns in dashboard

In [ ]:
# plan a: all platforms DOLocationID x data x hour

print('Aggregating by dropoff zone × date × hour...')

for year in [2024, 2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )

        group_cols = [
            'DOLocationID', 'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'do_hour', 'policy_period'
        ]
        df_DO_zone_hour = compute_metrics(df_month, group_cols) \
            .orderBy('DOLocationID', 'date', 'do_hour')

        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_DOzone_hour_{year}_{month:02d}.parquet')
        df_DO_zone_hour.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)

        df_month.unpersist()
        df_DO_zone_hour.unpersist()

In [ ]:
# plan b: classify by platforms (Uber and Lyft) DOLocationID x data x hour x platform

print('Aggregating by dropoff zone × date × hour x platform...')

def is_complete(output_file):
    success_marker = os.path.join(output_file, '_SUCCESS')
    return os.path.exists(success_marker)

for year in [2024, 2025]:
    for month in range(1, 13):
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_DOzone_hour_uber_{year}_{month:02d}.parquet')
        if is_complete(output_file):
            print(f'Already exists, skipping: {output_file}')
            continue
        
        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month) & (F.col('hvfhs_license_num') == 'HV0003') 
        )
        
        group_cols = [
            'DOLocationID', 'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'do_hour', 'policy_period', 'hvfhs_license_num'
        ]

        df_agg = compute_metrics(df_month, group_cols)
        df_agg = df_agg.orderBy('DOLocationID', 'date', 'do_hour')

        df_agg.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_month.unpersist()
        df_agg.unpersist()
        spark.catalog.clearCache()
        gc.collect()

In [ ]:
# plan b: classify by platforms (Uber and Lyft) DOLocationID x data x hour x platform

print('Aggregating by dropoff zone × date × hour x platform...')

def is_complete(output_file):
    success_marker = os.path.join(output_file, '_SUCCESS')
    return os.path.exists(success_marker)

for year in [2024, 2025]:
    for month in range(1, 13):
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_DOzone_hour_lyft_{year}_{month:02d}.parquet')
        if is_complete(output_file):
            print(f'Already exists, skipping: {output_file}')
            continue
        
        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month) & (F.col('hvfhs_license_num') == 'HV0005') 
        )
        
        group_cols = [
            'DOLocationID', 'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'do_hour', 'policy_period', 'hvfhs_license_num'
        ]

        df_agg = compute_metrics(df_month, group_cols)
        df_agg = df_agg.orderBy('DOLocationID', 'date', 'do_hour')

        df_agg.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_month.unpersist()
        df_agg.unpersist()
        spark.catalog.clearCache()
        gc.collect()

---
### 6.9 fhvhv_by_OD_month.parquet
Trip count and fare metrics aggregated by `PULocationID × DOLocationID × year × month`

> Core OD matrix — used for zone click heatmap and OD connectivity explorer in dashboard

**Note:** No wait time at OD pair level — wait time is a pickup-side metric only.

In [ ]:
print('Aggregating by OD pair × month...')

for year in [2024, 2025]:
    print(f'Processing year {year}...')
    df_year = df_feat.filter(F.col('year') == year)

    group_cols = [
        'PULocationID', 'DOLocationID',
        'year', 'month', 'policy_period'
    ]

    df_OD_month = df_year.groupBy(group_cols).agg(
        F.count('*').alias('trip_count'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('driver_pay').alias('_sum_driver_pay'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
        F.sum('cbd_congestion_fee').alias('total_cbd_fee')
    ).withColumn(
        'avg_base_fare',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_total_fare',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_driver_pay',   F.round(F.col('_sum_driver_pay') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_trip_miles',   F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4)
    ).drop(
        '_sum_base_fare', '_sum_total_fare', '_sum_driver_pay',
        '_sum_trip_miles', '_sum_trip_duration_min'
    ).orderBy('PULocationID', 'DOLocationID', 'year', 'month')

    output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_OD_month_{year}.parquet')
    df_OD_month.write.mode('overwrite').parquet(output_file)
    print(f'Saved: {output_file}')

    df_check = spark.read.parquet(output_file)
    print(f'Rows:            {df_check.count():,}')
    print(f'Unique OD pairs: {df_check.select("PULocationID", "DOLocationID").distinct().count():,}')
    df_check.show(5)

    df_year.unpersist()
    df_OD_month.unpersist()

---
### 6.10 fhvhv_by_OD_hour.parquet
Trip count aggregated by `PULocationID × DOLocationID × date × hour` (monthly files)

> Hourly OD flows — used for two-zone route comparison panel in dashboard

**Note:** Files are split by month to keep sizes manageable. Skip this section initially
if memory is a concern — only needed for hourly route comparison, not the zone click heatmap.

In [ ]:
print('Aggregating by OD pair × date × hour...')

for year in [2024, 2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )

        group_cols = [
            'PULocationID', 'DOLocationID',
            'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'hour', 'policy_period'
        ]

        df_OD_hour = df_month.groupBy(group_cols).agg(
            F.count('*').alias('trip_count'),
            F.sum('base_passenger_fare').alias('_sum_base_fare'),
            F.sum('total_fare').alias('_sum_total_fare'),
            F.sum('trip_miles').alias('_sum_trip_miles'),
            F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
            F.sum('cbd_congestion_fee').alias('total_cbd_fee')
        ).withColumn(
            'avg_base_fare',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_total_fare',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_trip_miles',   F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4)
        ).drop(
            '_sum_base_fare', '_sum_total_fare',
            '_sum_trip_miles', '_sum_trip_duration_min'
        ).orderBy('PULocationID', 'DOLocationID', 'date', 'hour')

        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_OD_hour_{year}_{month:02d}.parquet')
        df_OD_hour.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)

        df_month.unpersist()
        df_OD_hour.unpersist()

In [ ]:
print('Aggregating by OD pair × date × hour x platform...')

def is_complete(output_file):
    success_marker = os.path.join(output_file, '_SUCCESS')
    return os.path.exists(success_marker)

for year in [2024, 2025]:
    for month in range(1, 13):
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_OD_hour_uber_{year}_{month:02d}.parquet')
        if is_complete(output_file):
            print(f'Already exists, skipping: {output_file}')
            continue

        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month) & (F.col('hvfhs_license_num') == 'HV0003')
        )
        group_cols = [
            'PULocationID', 'DOLocationID',
            'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'hour', 'policy_period', 'hvfhs_license_num'
        ]
        df_OD_hour = df_month.groupBy(group_cols).agg(
            F.count('*').alias('trip_count'),
            F.sum('base_passenger_fare').alias('_sum_base_fare'),
            F.sum('total_fare').alias('_sum_total_fare'),
            F.sum('trip_miles').alias('_sum_trip_miles'),
            F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
            F.sum('cbd_congestion_fee').alias('total_cbd_fee')
        ).withColumn(
            'avg_base_fare',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_total_fare',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_trip_miles',   F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4)
        ).drop(
            '_sum_base_fare', '_sum_total_fare',
            '_sum_trip_miles', '_sum_trip_duration_min'
        ).orderBy('PULocationID', 'DOLocationID', 'date', 'hour')

        df_OD_hour.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_month.unpersist()
        df_OD_hour.unpersist()
        spark.catalog.clearCache()
        gc.collect()

In [ ]:
print('Aggregating by OD pair × date × hour x platform...')

def is_complete(output_file):
    success_marker = os.path.join(output_file, '_SUCCESS')
    return os.path.exists(success_marker)

for year in [2024, 2025]:
    for month in range(1, 13):
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_OD_hour_lyft_{year}_{month:02d}.parquet')
        if is_complete(output_file):
            print(f'Already exists, skipping: {output_file}')
            continue

        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month) & (F.col('hvfhs_license_num') == 'HV0005')
        )
        group_cols = [
            'PULocationID', 'DOLocationID',
            'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'hour', 'policy_period', 'hvfhs_license_num'
        ]
        df_OD_hour = df_month.groupBy(group_cols).agg(
            F.count('*').alias('trip_count'),
            F.sum('base_passenger_fare').alias('_sum_base_fare'),
            F.sum('total_fare').alias('_sum_total_fare'),
            F.sum('trip_miles').alias('_sum_trip_miles'),
            F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
            F.sum('cbd_congestion_fee').alias('total_cbd_fee')
        ).withColumn(
            'avg_base_fare',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_total_fare',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_trip_miles',   F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4)
        ).drop(
            '_sum_base_fare', '_sum_total_fare',
            '_sum_trip_miles', '_sum_trip_duration_min'
        ).orderBy('PULocationID', 'DOLocationID', 'date', 'hour')

        df_OD_hour.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_month.unpersist()
        df_OD_hour.unpersist()
        spark.catalog.clearCache()
        gc.collect()

---
## 6.11 — Daily Citywide Aggregation

### fhvhv_by_date.parquet
All metrics by `date × hvfhs_license_num`

> Single combined 2024-2025 file — ~1,500 rows, very fast to query.
> Used for citywide daily trend chart in dashboard.
> Jan 1-4 2025 excluded (grey zone).

In [ ]:
print('Aggregating citywide daily × platform...')

# # Exclude Jan 1-4 2025 grey zone
# df_daily = df_feat.filter(
#     ~(
#         (F.col('year') == 2025) &
#         (F.col('month') == 1) &
#         (F.col('date') < F.lit('2025-01-05'))
#     )
# )


df_daily = df_feat

group_cols = [
    'date', 'year', 'month', 'week',
    'day_of_week', 'is_weekend',
    'hvfhs_license_num', 'policy_period'
]

df_by_date = compute_metrics(df_daily, group_cols) \
    .orderBy('date', 'hvfhs_license_num')

output_file = os.path.join(OUTPUT_PATH, 'fhvhv_by_date.parquet')
df_by_date.write.mode('overwrite').parquet(output_file)
print(f'Saved: {output_file}')

df_check = spark.read.parquet(output_file)
print(f'Rows: {df_check.count():,}')
df_check.show(5)

df_daily.unpersist()
df_by_date.unpersist()

---
## 6.12 — Daily Pickup Zone Aggregation

### fhvhv_by_zone_date.parquet
All metrics by `PULocationID × date × hvfhs_license_num`

> Single combined 2024-2025 file.
> Used for origin daily choropleth map in dashboard.
> Jan 1-4 2025 excluded (grey zone).

In [ ]:
print('Aggregating by PU zone × date × platform...')

## Exclude Jan 1-4 2025 grey zone
# df_daily = df_feat.filter(
#     ~(
#         (F.col('year') == 2025) &
#         (F.col('month') == 1) &
#         (F.col('date') < F.lit('2025-01-05'))
#     )
# )

df_daily = df_feat

group_cols = [
    'PULocationID', 'date', 'year', 'month', 'week',
    'day_of_week', 'is_weekend',
    'hvfhs_license_num', 'policy_period'
]

df_zone_date = compute_metrics(df_daily, group_cols) \
    .orderBy('PULocationID', 'date', 'hvfhs_license_num')

output_file = os.path.join(OUTPUT_PATH, 'fhvhv_by_zone_date.parquet')
df_zone_date.write.mode('overwrite').parquet(output_file)
print(f'Saved: {output_file}')

df_check = spark.read.parquet(output_file)
print(f'Rows: {df_check.count():,}')
df_check.show(5)

df_daily.unpersist()
df_zone_date.unpersist()

---
## 6.13 — Daily Dropoff Zone Aggregation

### fhvhv_by_DOzone_date.parquet
All metrics by `DOLocationID × date × hvfhs_license_num`

> Single combined 2024-2025 file.
> Used for destination daily choropleth map in dashboard.
> Jan 1-4 2025 excluded (grey zone).

In [ ]:
print('Aggregating by DO zone × date × platform...')

# # Exclude Jan 1-4 2025 grey zone
# df_daily = df_feat.filter(
#     ~(
#         (F.col('year') == 2025) &
#         (F.col('month') == 1) &
#         (F.col('date') < F.lit('2025-01-05'))
#     )
# )

df_daily = df_feat

group_cols = [
    'DOLocationID', 'date', 'year', 'month', 'week',
    'day_of_week', 'is_weekend',
    'hvfhs_license_num', 'policy_period'
]

df_DOzone_date = compute_metrics(df_daily, group_cols) \
    .orderBy('DOLocationID', 'date', 'hvfhs_license_num')

output_file = os.path.join(OUTPUT_PATH, 'fhvhv_by_DOzone_date.parquet')
df_DOzone_date.write.mode('overwrite').parquet(output_file)
print(f'Saved: {output_file}')

df_check = spark.read.parquet(output_file)
print(f'Rows: {df_check.count():,}')
df_check.show(5)

df_daily.unpersist()
df_DOzone_date.unpersist()

---
## 6.14 — Daily OD Aggregation

### fhvhv_by_OD_date_{year}.parquet
Trip count + fares by `PULocationID × DOLocationID × date × hvfhs_license_num`

> Split by year to keep file sizes manageable.
> **Filter: combined Uber + Lyft trips ≥ 5 per day** per OD pair.
> Used for zone click OD heatmap and OD explorer in dashboard.
> Jan 1-4 2025 excluded (grey zone).

**Two-step filter logic:**
1. Aggregate all OD pairs per day per platform
2. Sum across platforms → keep OD × date combos with combined total ≥ 5
3. Join back to retain both Uber and Lyft rows for kept pairs

In [ ]:
print('Aggregating by OD pair × date × platform (combined >= 5 filter)...')

# # Exclude Jan 1-4 2025 grey zone
# df_daily = df_feat.filter(
#     ~(
#         (F.col('year') == 2025) &
#         (F.col('month') == 1) &
#         (F.col('date') < F.lit('2025-01-05'))
#     )
# )

df_daily = df_feat

for year in [2024, 2025]:
    print(f'Processing year {year}...')
    df_year = df_daily.filter(F.col('year') == year)

    # Step 1: aggregate per OD pair x date x platform
    group_cols = [
        'PULocationID', 'DOLocationID',
        'date', 'year', 'month', 'week',
        'day_of_week', 'is_weekend',
        'hvfhs_license_num', 'policy_period'
    ]

    df_OD_raw = df_year.groupBy(group_cols).agg(
        F.count('*').alias('trip_count'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('driver_pay').alias('_sum_driver_pay'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
        F.sum('cbd_congestion_fee').alias('total_cbd_fee')
    ).withColumn(
        'avg_base_fare',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_total_fare',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_driver_pay',   F.round(F.col('_sum_driver_pay') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_trip_miles',   F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4)
    ).withColumn(
        'avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4)
    ).drop(
        '_sum_base_fare', '_sum_total_fare', '_sum_driver_pay',
        '_sum_trip_miles', '_sum_trip_duration_min'
    )

    # Step 2: compute combined daily total per OD pair x date
    df_combined = df_OD_raw.groupBy(
        'PULocationID', 'DOLocationID', 'date'
    ).agg(
        F.sum('trip_count').alias('combined_daily_trips')
    ).filter(
        F.col('combined_daily_trips') >= 5
    ).select('PULocationID', 'DOLocationID', 'date')

    # Step 3: join back to keep only significant OD pairs
    df_OD_date = df_OD_raw.join(
        df_combined,
        on=['PULocationID', 'DOLocationID', 'date'],
        how='inner'
    ).orderBy('PULocationID', 'DOLocationID', 'date', 'hvfhs_license_num')

    output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_OD_date_{year}.parquet')
    df_OD_date.write.mode('overwrite').parquet(output_file)
    print(f'Saved: {output_file}')

    df_check = spark.read.parquet(output_file)
    total_rows   = df_check.count()
    unique_pairs = df_check.select('PULocationID', 'DOLocationID').distinct().count()
    unique_od_dates = df_check.select('PULocationID', 'DOLocationID', 'date').distinct().count()
    print(f'  Total rows:            {total_rows:,}')
    print(f'  Unique OD pairs:       {unique_pairs:,}')
    print(f'  Unique OD-date combos: {unique_od_dates:,}')
    df_check.show(5)

    df_year.unpersist()
    df_OD_raw.unpersist()
    df_OD_date.unpersist()

df_daily.unpersist()

---
## 6.15 — Platform Coverage Validation

Sanity check — confirms Uber (HV0003) and Lyft (HV0005) are both
present and correctly split across all new platform-dimension files.

In [ ]:
print('=' * 65)
print('PLATFORM COVERAGE VALIDATION')
print('=' * 65)

PLATFORM_MAP = {'HV0003': 'Uber', 'HV0005': 'Lyft'}

files_to_check = [
    ('fhvhv_by_date.parquet',              'Citywide daily'),
    ('fhvhv_by_zone_date.parquet',         'PU zone daily'),
    ('fhvhv_by_DOzone_date.parquet',       'DO zone daily'),
    ('fhvhv_by_OD_date_2024.parquet',      'OD daily 2024'),
    ('fhvhv_by_OD_date_2025.parquet',      'OD daily 2025'),
]

print(f'{"File":<30} {"Uber trips":>15} {"Lyft trips":>15} {"Total":>15}')
print('-' * 78)

for fname, label in files_to_check:
    path = os.path.join(OUTPUT_PATH, fname)
    if not os.path.exists(path):
        print(f'{label:<30} FILE NOT FOUND')
        continue
    df_tmp = spark.read.parquet(path)
    rows = df_tmp.groupBy('hvfhs_license_num') \
                 .agg(F.sum('trip_count').alias('trips')) \
                 .collect()
    counts = {PLATFORM_MAP.get(r['hvfhs_license_num'], r['hvfhs_license_num']): r['trips'] for r in rows}
    uber  = counts.get('Uber', 0)
    lyft  = counts.get('Lyft', 0)
    total = uber + lyft
    print(f'{label:<30} {uber:>15,.0f} {lyft:>15,.0f} {total:>15,.0f}')

print()
print('Both Uber and Lyft should appear in every file above.')
print('Uber market share should be ~70-75% based on known TLC data.')

## 7. Summary

In [ ]:
print('=' * 65)
print('PROCESSING COMPLETE — OUTPUT SUMMARY')
print('=' * 65)

output_files = [
    # Existing pickup-side files
    ('fhvhv_by_zone_month_{year}.parquet',        'PU zone × month',            'All metrics'),
    ('fhvhv_by_hour_{year}.parquet',              'Date × hour',                'All metrics'),
    ('fhvhv_by_zone_hour_{year}_{month}.parquet', 'PU zone × date × hour',      'All metrics'),
    ('fhvhv_by_15min_{year}_{month}.parquet',     'Date × 15min',               'Wait time only'),
    ('fhvhv_by_zone_15min_{year}_{month}.parquet','PU zone × date × 15min',     'Wait time only'),
    # New dropoff-side files
    ('fhvhv_by_DOzone_month_{year}.parquet',      'DO zone × month',            'All metrics'),
    ('fhvhv_by_DOzone_hour_{year}_{month}.parquet','DO zone × date × hour',     'All metrics'),
    # New OD pair files
    ('fhvhv_by_OD_month_{year}.parquet',          'PU × DO zone × month',       'Trip count + fares'),
    ('fhvhv_by_OD_hour_{year}_{month}.parquet',   'PU × DO zone × date × hour', 'Trip count + fares'),
]

print(f'\n{"File pattern":<52} {"Granularity":<30} {"Metrics"}')
print('-' * 100)
for fname, granularity, metrics in output_files:
    print(f'{fname:<52} {granularity:<30} {metrics}')

# Check actual files on disk
print('\n' + '=' * 65)
print('FILES ON DISK')
print('=' * 65)
import glob as _glob
all_outputs = sorted(_glob.glob(os.path.join(OUTPUT_PATH, "*.parquet")))
total_size_mb = 0
for path in all_outputs:
    if os.path.isdir(path):
        size_mb = sum(
            os.path.getsize(os.path.join(path, f))
            for f in os.listdir(path) if f.endswith(".parquet")
        ) / (1024 * 1024)
    else:
        size_mb = os.path.getsize(path) / (1024 * 1024)
    total_size_mb += size_mb
    print(f'  {os.path.basename(path):<57} {size_mb:>8.1f} MB')
print(f'\n  {"Total output size":<57} {total_size_mb:>8.1f} MB')

# Row counts
print('\n' + '=' * 65)
print('ROW COUNTS')
print('=' * 65)
print(f'  Raw rows:      {raw_count:>15,}')
print(f'  Cleaned rows:  {clean_count:>15,}')
print(f'  Rows removed:  {raw_count - clean_count:>15,}  ({(raw_count - clean_count)/raw_count*100:.2f}%)')
print('\n' + '=' * 65)
print('Next step: Data validation → Dashboard build')
print('=' * 65)


---
## Notes

**Cleaning rules applied:**
- Removed null pickup/dropoff/request datetimes
- Removed trips with `trip_miles <= 0` or `trip_time <= 0`
- Removed trips with `base_passenger_fare <= 0`
- Removed trips where `pickup_datetime < request_datetime`
- Removed extreme outliers: `trip_miles > 200`, `trip_time > 10800s`
- Filled null `cbd_congestion_fee` with 0 (pre-policy trips)

**Aggregation method:** Aggregate-first (sum numerators and denominators separately before dividing)

**Output files:**
| File | Granularity | Metrics |
|---|---|---|
| `fhvhv_by_zone_month.parquet` | Zone × Month | All metrics |
| `fhvhv_by_hour.parquet` | Date × Hour | All metrics |
| `fhvhv_by_zone_hour.parquet` | Zone × Date × Hour | All metrics |
| `fhvhv_by_15min.parquet` | Date × 15min | Wait time only |
| `fhvhv_by_zone_15min.parquet` | Zone × Date × 15min | Wait time only |